# 02 Preprocessing

This notebook prepares clean, model-ready datasets and saves them to data/processed.

In [ ]:
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

root_dir = Path.cwd()
if not (root_dir / "src").exists() and (root_dir.parent / "src").exists():
    root_dir = root_dir.parent

if not (root_dir / "src").exists():
    raise FileNotFoundError("Could not find project root containing 'src' directory")

sys.path.insert(0, str(root_dir))

from src.data_loader import load_all_raw_datasets
from src.preprocess import (
    clean_dataframe,
    encode_categoricals,
    identify_effort_column,
    scale_numeric_features,
    save_multiple_processed,
    split_features_target,
    )

raw_datasets = load_all_raw_datasets(root_dir / "data" / "raw")
processed_output = {}

In [2]:
for name, df in raw_datasets.items():
    effort_col = identify_effort_column(df)
    cleaned = clean_dataframe(df)
    encoded = encode_categoricals(cleaned)

    if effort_col not in encoded.columns:
        effort_col = encoded.columns[-1]

    x, y = split_features_target(encoded, effort_col)
    x_scaled, _ = scale_numeric_features(x)
    processed_df = x_scaled.copy()
    processed_df[effort_col] = pd.to_numeric(y, errors="coerce")
    processed_df = processed_df.dropna(subset=[effort_col])
    processed_output[name] = processed_df

    print(f"{name}: raw={df.shape}, cleaned={cleaned.shape}, processed={processed_df.shape}")

cocomo81: raw=(63, 19), cleaned=(63, 19), processed=(63, 20)
desharnais: raw=(81, 13), cleaned=(81, 13), processed=(81, 13)
china: raw=(499, 19), cleaned=(499, 19), processed=(499, 19)


In [3]:
saved_paths = save_multiple_processed(processed_output, root_dir / "data" / "processed")
print("Saved files:")
for name, path in saved_paths.items():
    print(f"- {name}: {path}")

Saved files:
- cocomo81: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\data\processed\cocomo81_processed.csv
- desharnais: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\data\processed\desharnais_processed.csv
- china: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\data\processed\china_processed.csv
